# Maia Lite Chat 0.3.0 — Multilingual Corpus Quality Pilot v8.4

**Colab notebook · quality/stratification analysis only · no training export**

This notebook complements, rather than replaces, the validated v8.3 acquisition pilot.

**Goals**
- Reuse `pilot_stats.json` from v8.3 and independently measure bounded samples from **FineWeb2 PT/ES, Wikipedia PT/EN/ES, and FineWeb-Edu EN**.
- Verify configurations at runtime, and **skip invalid configurations explicitly**.
- Record language identification agreement, text length, duplication and near-duplication **within this pilot only**.
- Inspect OpenAssistant OASST1 language labels and reconstruct complete **observed parent chains** (not assume that isolated messages are dialogues).
- Produce per-source diagnostics and planning estimates, **without extrapolating sampled volume to whole datasets**.
- Keep rights review pending; do not export training data or assume licenses imply approval.

The pilot samples a bounded **prefix window**, using deterministic reservoir sampling *within that window*. This improves over using the first 150 records, but **does not establish dataset-wide statistical representativeness**. To estimate corpus-wide yield, later sample randomized shards/partitions with known sampling probabilities.


**Correção de compatibilidade (18/10/2026):** removido `trust_remote_code=False` das chamadas `get_dataset_config_names`, `get_dataset_split_names` e `load_dataset` para evitar o erro `DownloadConfig.__init__() got an unexpected keyword argument`. A política de não executar código remoto permanece a padrão da versão instalada de `datasets`; não habilite `trust_remote_code=True`. Esta é a v8.4 corrigida, sem novo pipeline.


In [ ]:
%pip -q install "datasets>=3.0,<5" "huggingface_hub>=0.26" "pandas>=2.0" "langid>=1.1.6" "numpy>=1.24" 

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import json, hashlib, random, re, time, statistics, math
import pandas as pd
import langid
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names
from huggingface_hub import HfApi

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Running outside Google Colab")

ROOT = Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat")
PREVIOUS_RUN = ROOT / "corpus_v8/runs/acquisition_pilot_v8_3_001"
OUTPUT = ROOT / "corpus_v8/runs/quality_pilot_v8_4_001"
OUTPUT.mkdir(parents=True, exist_ok=True)

# Conservative pilot budgets: change explicitly if necessary.
WINDOW_ROWS = 2000                  # prefix scanned per source, not the entire dataset
RESERVOIR_SIZE = 250                # sampled documents per source
OASST_SCAN_ROWS = 5000              # enough to observe some parent relationships
MAX_TEXT_CHARS = 20000              # only used for analysis, not export
MAX_WALL_SECONDS_PER_JOB = 240      # checked between yielded records (network stalls can exceed this)
MIN_CHARS = 100
SEED = 20261018
SAVE_TEXT_EXCERPTS = False          # default: no raw text saved
EXCERPT_CHARS = 400
RUN_STREAMING = True                # set False to only analyze existing v8.3 stats
langid.set_languages(["pt","en","es"])
print("Previous:", PREVIOUS_RUN)
print("Output:", OUTPUT)

## 1. Read previous pilot results (v8.3)

If the earlier file is unavailable, the new pilot still runs. Do not treat the old 3,036,518 character-based proxy tokens as true tokenizer tokens.


In [ ]:
previous_path = PREVIOUS_RUN / "pilot_stats.json"
previous = json.loads(previous_path.read_text(encoding="utf-8")) if previous_path.exists() else []
if previous:
    old = pd.DataFrame(previous)
    display(old[["job_id","status","rows_sampled","proxy_tokens"]])
    print("Old proxy-token total:", sum(int(x.get("proxy_tokens",0)) for x in previous))
else:
    print("Previous v8.3 stats not found; continuing with fresh pilot.")

## 2. Explicit acquisition jobs

English is **not** requested from FineWeb2. FineWeb-Edu `sample-10BT` is checked against the current dataset configurations at runtime. This notebook makes no assumption that a dataset's config names are immutable.


In [ ]:
JOBS = [
    {"id":"fineweb2_pt","dataset":"HuggingFaceFW/fineweb-2","config":"por_Latn","expected":"pt","family":"common_crawl"},
    {"id":"fineweb2_es","dataset":"HuggingFaceFW/fineweb-2","config":"spa_Latn","expected":"es","family":"common_crawl"},
    {"id":"fineweb_edu_en","dataset":"HuggingFaceFW/fineweb-edu","config":"sample-10BT","expected":"en","family":"common_crawl"},
    {"id":"wikipedia_pt","dataset":"wikimedia/wikipedia","config":"20231101.pt","expected":"pt","family":"wikipedia"},
    {"id":"wikipedia_en","dataset":"wikimedia/wikipedia","config":"20231101.en","expected":"en","family":"wikipedia"},
    {"id":"wikipedia_es","dataset":"wikimedia/wikipedia","config":"20231101.es","expected":"es","family":"wikipedia"},
]
api = HfApi()
checks = []
for dataset in sorted(set(j["dataset"] for j in JOBS) | {"OpenAssistant/oasst1"}):
    record = {"dataset":dataset,"metadata_ok":False,"sha":None,"license_claim":None,"configs":[],
              "configs_status":"unknown","error":None}
    try:
        info = api.dataset_info(dataset)
        record["metadata_ok"]=True
        record["sha"]=info.sha
        record["license_claim"]=(info.cardData or {}).get("license")
    except Exception as e:
        record["error"]=f"metadata: {type(e).__name__}: {str(e)[:180]}"
    try:
        record["configs"]=get_dataset_config_names(dataset)
        record["configs_status"]="ok"
    except Exception as e:
        record["configs_status"]="error"
        record["error"]=(record["error"] or "")+f" configs: {type(e).__name__}: {str(e)[:180]}"
    checks.append(record)
check_by_id={r["dataset"]:r for r in checks}
for job in JOBS:
    r=check_by_id[job["dataset"]]
    job["config_verified"]=job["config"] in r["configs"]
    print(job["id"],"config available:",job["config_verified"],
          "metadata:",r["metadata_ok"],"configs status:",r["configs_status"])
(OUTPUT/"source_config_checks.json").write_text(
    json.dumps(checks,ensure_ascii=False,indent=2,default=str),encoding="utf-8")

## 3. Bounded sampling and quality features

Language identification is a **heuristic**. We use `langid` constrained to PT/EN/ES, so documents in other languages can be misclassified as one of those three. Low confidence and disagreement require manual audit. We avoid recording full text by default.

Exact hashes and SimHash comparisons are scoped to this sample. They cannot prove that the full corpus has been deduplicated.


In [ ]:
def extract_text(row):
    for key in ("text","content","body","document","raw_content"):
        val=row.get(key)
        if isinstance(val,str) and val.strip():
            return val,key
    return None,None

def normalize(text):
    return re.sub(r"\s+"," ",text.casefold()).strip()

def proxy_tokens(text):
    return max(1,round(len(text)/4))

def simhash64(text):
    words=re.findall(r"\w+",normalize(text))
    shingles=[" ".join(words[i:i+5]) for i in range(max(0,len(words)-4))]
    if not shingles:
        shingles=words
    weights=[0]*64
    # bounded to limit CPU on long documents
    for token in shingles[:1500]:
        digest=int.from_bytes(hashlib.blake2b(token.encode("utf-8"),digest_size=8).digest(),"big")
        for bit in range(64):
            weights[bit]+=1 if (digest>>bit)&1 else -1
    return sum((1<<i) for i,v in enumerate(weights) if v>0)

def features(text,job,idx,field):
    clipped=text[:MAX_TEXT_CHARS]
    pred, score=langid.classify(clipped[:3000])
    result={
        "job_id":job["id"],"dataset":job["dataset"],"family":job["family"],
        "expected_language":job["expected"],"predicted_language":pred,
        "langid_score":round(float(score),3),
        "row_index_in_prefix":idx,"field":field,
        "chars":len(text),"proxy_tokens":proxy_tokens(text),
        "words":len(re.findall(r"\w+",text)),
        "lines":text.count("\n")+1,
        "url_like":bool(re.search(r"https?://",text)),
        "replacement_char_count":text.count("\ufffd"),
        "nonprintable_count":sum(1 for ch in text[:5000] if ord(ch)<32 and ch not in "\n\t\r"),
        "normalized_sha256":hashlib.sha256(normalize(text).encode("utf-8")).hexdigest(),
        "simhash64":f"{simhash64(text):016x}",
        "rights_status":"pending_review",
        "eligible_for_training":False
    }
    if SAVE_TEXT_EXCERPTS:result["text_excerpt"]=text[:EXCERPT_CHARS]
    return result

def stream_prefix_reservoir(job, seed):
    result={"job_id":job["id"],"status":"not_run","config":job["config"],
            "split":None,"rows_scanned":0,"rows_valid":0,"rows_sampled":0,
            "error":None,"window_rows":WINDOW_ROWS}
    if not RUN_STREAMING:
        result["status"]="disabled";return result,[]
    if not job["config_verified"]:
        result["status"]="config_not_verified";return result,[]
    try:
        splits=get_dataset_split_names(job["dataset"],config_name=job["config"])
        split="train" if "train" in splits else (splits[0] if splits else None)
        if split is None:raise ValueError("No split found")
        result["split"]=split
        ds=load_dataset(job["dataset"],name=job["config"],split=split,
                        streaming=True)
        rng=random.Random(seed)
        reservoir=[]
        started=time.monotonic()
        for idx,row in enumerate(ds):
            if idx>=WINDOW_ROWS:
                break
            if time.monotonic()-started>MAX_WALL_SECONDS_PER_JOB:
                result["status"]="time_budget_reached";break
            result["rows_scanned"]+=1
            text,field=extract_text(row)
            if not text or len(text)<MIN_CHARS:
                continue
            result["rows_valid"]+=1
            # reservoir within bounded prefix; features computed only for selected rows
            entry=(idx,text,field)
            if len(reservoir)<RESERVOIR_SIZE:
                reservoir.append(entry)
            else:
                j=rng.randrange(result["rows_valid"])
                if j<RESERVOIR_SIZE:reservoir[j]=entry
        features_list=[features(t,job,i,field) for i,t,field in reservoir]
        result["rows_sampled"]=len(features_list)
        if result["status"]=="not_run":result["status"]="sampled"
        return result,features_list
    except Exception as e:
        result["status"]="error"
        result["error"]=f"{type(e).__name__}: {str(e)[:400]}"
        return result,[]

job_results=[]
documents=[]
for i,job in enumerate(JOBS):
    result,items=stream_prefix_reservoir(job,SEED+i)
    job_results.append(result)
    documents.extend(items)
    print(job["id"],result["status"],"scanned",result["rows_scanned"],
          "sampled",result["rows_sampled"],"error",result["error"])
(OUTPUT/"job_statuses.json").write_text(json.dumps(job_results,indent=2,ensure_ascii=False),encoding="utf-8")
with (OUTPUT/"document_features.jsonl").open("w",encoding="utf-8") as f:
    for row in documents:f.write(json.dumps(row,ensure_ascii=False)+"\n")
print("Feature rows:",len(documents),"raw excerpts saved:",SAVE_TEXT_EXCERPTS)

## 4. Diagnostics by source and language

Do not equate the measured language share of equal-size source samples with the language mix of the intended training corpus. The actual training mixture must be sampled deliberately.


In [ ]:
def hamming64(a,b):
    return (int(a,16)^int(b,16)).bit_count()

def summarize_source(rows):
    if not rows:return {}
    lengths=sorted(r["chars"] for r in rows)
    hashes=Counter(r["normalized_sha256"] for r in rows)
    exact_dup=sum(n-1 for n in hashes.values() if n>1)
    # Approximate duplicates: compare same-source documents only, with length ratio constraint.
    # O(n^2) is safe at the bounded pilot size; do not use this for full corpus.
    near_pairs=0
    for i in range(len(rows)):
        for j in range(i+1,len(rows)):
            a,b=rows[i],rows[j]
            if a["normalized_sha256"]==b["normalized_sha256"]:continue
            if min(a["chars"],b["chars"])/max(a["chars"],b["chars"])<0.75:continue
            if hamming64(a["simhash64"],b["simhash64"])<=3:near_pairs+=1
    return {
        "sampled":len(rows),
        "proxy_tokens":sum(r["proxy_tokens"] for r in rows),
        "median_chars":statistics.median(lengths),
        "p90_chars":lengths[min(len(lengths)-1,int(0.9*(len(lengths)-1)))],
        "language_agreement_pct":round(100*sum(r["predicted_language"]==r["expected_language"] for r in rows)/len(rows),1),
        "predicted_languages":dict(Counter(r["predicted_language"] for r in rows)),
        "exact_duplicate_extra_docs":exact_dup,
        "near_duplicate_pairs_sample_only":near_pairs,
        "replacement_char_docs":sum(r["replacement_char_count"]>0 for r in rows),
        "nonprintable_docs":sum(r["nonprintable_count"]>0 for r in rows),
        "url_like_docs":sum(r["url_like"] for r in rows),
    }

by_job={job["id"]:[r for r in documents if r["job_id"]==job["id"]] for job in JOBS}
summaries=[{"job_id":job["id"],"expected":job["expected"],
            **summarize_source(by_job[job["id"]])} for job in JOBS]
df_summary=pd.DataFrame(summaries)
display(df_summary)
df_summary.to_csv(OUTPUT/"quality_summary.csv",index=False)
(OUTPUT/"quality_summary.json").write_text(json.dumps(summaries,indent=2,ensure_ascii=False),encoding="utf-8")


## 5. OpenAssistant: language counts and observed dialogue chains

OASST1 stores **messages and parent references**. A sampled prefix may omit parents and cannot establish the complete conversation tree. We count languages from available metadata and reconstruct only **observed** parent chains. A chain is marked complete only if it reaches a root present in the scan.

This does **not** turn OASST1 into an approved training corpus. No conversation text is saved.


In [ ]:
oasst_result={"status":"not_run","scanned":0,"messages_with_text":0,"languages":{},
              "root_messages":0,"parent_references":0,"complete_chains":0,
              "incomplete_chains":0,"max_observed_chain_length":0,
              "roles":{},"error":None}
oasst_examples=[]
if not RUN_STREAMING:
    oasst_result["status"]="disabled"
else:
    try:
        ds=load_dataset("OpenAssistant/oasst1",split="train",streaming=True)
        observed={}
        langs=Counter();roles=Counter()
        start=time.monotonic()
        for idx,row in enumerate(ds):
            if idx>=OASST_SCAN_ROWS or time.monotonic()-start>MAX_WALL_SECONDS_PER_JOB:
                break
            oasst_result["scanned"]+=1
            text=row.get("text")
            if not isinstance(text,str) or not text.strip():continue
            oasst_result["messages_with_text"]+=1
            lang=row.get("lang") or row.get("language") or "unknown"
            langs[str(lang)]+=1
            role=row.get("role") or ("assistant" if row.get("role")=="assistant" else "unknown")
            roles[str(role)]+=1
            mid=row.get("message_id")
            parent=row.get("parent_id")
            if mid is not None:
                observed[str(mid)]={"parent":str(parent) if parent is not None else None,
                                    "lang":str(lang),"role":str(role)}
        oasst_result["languages"]=dict(langs)
        oasst_result["roles"]=dict(roles)
        oasst_result["root_messages"]=sum(x["parent"] is None for x in observed.values())
        oasst_result["parent_references"]=sum(x["parent"] is not None for x in observed.values())
        complete=0;incomplete=0;max_depth=0
        for mid in observed:
            cur=mid;seen=set();depth=0
            while cur in observed and cur not in seen:
                seen.add(cur);depth+=1
                par=observed[cur]["parent"]
                if par is None:
                    complete+=1;break
                cur=par
            else:
                incomplete+=1
            max_depth=max(max_depth,depth)
        oasst_result.update(status="sampled",complete_chains=complete,
                            incomplete_chains=incomplete,max_observed_chain_length=max_depth)
    except Exception as e:
        oasst_result.update(status="error",error=f"{type(e).__name__}: {str(e)[:400]}")
print(json.dumps(oasst_result,indent=2,ensure_ascii=False))
(OUTPUT/"oasst_dialogue_diagnostics.json").write_text(
    json.dumps(oasst_result,indent=2,ensure_ascii=False),encoding="utf-8")

## 6. Pilot-only overlap across sources

This is **not** a full near-duplicate audit. It checks exact normalized-text hashes across the bounded sample and reports cross-family overlap when observed.


In [ ]:
seen=defaultdict(list)
for r in documents:
    seen[r["normalized_sha256"]].append((r["job_id"],r["family"]))
cross_job=[v for v in seen.values() if len({x[0] for x in v})>1]
cross_family=[v for v in seen.values() if len({x[1] for x in v})>1]
overlap={"sample_docs":len(documents),"cross_job_exact_duplicate_groups":len(cross_job),
         "cross_family_exact_duplicate_groups":len(cross_family),
         "note":"Absence in bounded samples does not imply corpus-wide absence."}
print(overlap)
(OUTPUT/"overlap_summary.json").write_text(json.dumps(overlap,indent=2),encoding="utf-8")

## 7. Acquisition planning manifest and blockers

Target **processed** tokens = `354,823,168 × 20 × 3 = 21,289,390,080`. Unique, reviewed, deduplicated corpus size is **unknown** until large-scale collection audits. This notebook does not assert production readiness or estimate total available dataset tokens from a prefix sample.


In [ ]:
PARAMETERS=354_823_168
TARGET_PROCESSED_TOKENS=PARAMETERS*20*3
TARGET_LANG_SHARES={"pt":0.40,"en":0.35,"es":0.25}
manifest={
    "version":"v8.4",
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "previous_run":str(PREVIOUS_RUN),
    "output_dir":str(OUTPUT),
    "target_processed_tokens":TARGET_PROCESSED_TOKENS,
    "target_processed_tokens_by_language":{k:round(v*TARGET_PROCESSED_TOKENS) for k,v in TARGET_LANG_SHARES.items()},
    "pilot_design":{"window_rows":WINDOW_ROWS,"reservoir_size":RESERVOIR_SIZE,
                    "sample_scope":"random reservoir within dataset prefix, not random dataset-wide shards",
                    "seed":SEED,"save_text_excerpts":SAVE_TEXT_EXCERPTS},
    "jobs":job_results,
    "oasst_status":oasst_result["status"],
    "oasst_languages":oasst_result["languages"],
    "pilot_document_count":len(documents),
    "pilot_proxy_tokens":sum(x["proxy_tokens"] for x in documents),
    "unique_approved_training_tokens":0,
    "full_corpus_unique_tokens_estimated":None,
    "rights_review_complete":False,
    "benchmark_frozen":False,
    "production_ready":False,
    "blockers":[
        "Rights and license reviews incomplete (source and document levels)",
        "Prefix-window sampling is not dataset-wide representative",
        "True Maia tokenizer counts unavailable",
        "No validated estimate of corpus-wide unique tokens",
        "No source-family-wide near-duplicate audit",
        "No frozen benchmark and contamination audit",
        "OASST observed parent chains may be incomplete",
        "Final tokenizer not trained/certified"
    ],
    "outputs":["source_config_checks.json","job_statuses.json","document_features.jsonl",
               "quality_summary.csv","quality_summary.json",
               "oasst_dialogue_diagnostics.json","overlap_summary.json",
               "manifest_quality_v8_4.json"]
}
manifest_path=OUTPUT/"manifest_quality_v8_4.json"
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
print("Manifest:",manifest_path)
print("Production ready:",manifest["production_ready"])
print("Pilot documents:",manifest["pilot_document_count"])
print("Pilot proxy tokens:",manifest["pilot_proxy_tokens"])
print("Errors:",{j["job_id"]:j["error"] for j in job_results if j["error"]})
print("OASST languages:",manifest["oasst_languages"])

## 8. Interpretation and next acquisition gate

**Please share:** `quality_summary.csv`, `oasst_dialogue_diagnostics.json`, and `manifest_quality_v8_4.json`.

**Before bulk acquisition**:
1. Verify license/rights for each source and its underlying content; maintain a documented human review.
2. Establish a *shard-aware* random sampling plan for FineWeb/FineWeb2 (not just the prefix).
3. Audit real PT/EN/ES proportions, contamination and duplicates using the **final Maia tokenizer**.
4. Keep authentic human dialogue distinct from synthetic dialogue.
5. Establish a frozen evaluation suite, with source/version hashes, prior to using candidate training documents.

**Safety:** The notebook stores only hashes, lengths, classifications and optional short excerpts (off by default); it does not export text for training.
